# Drive Workbench | Kaggriculture 3-pattern Full Logs v0.2

[Open in Colab](https://colab.research.google.com/github/kw0809suzuki-oss/relation-flow-agent/blob/main/experiments/drive_workbench/Kaggriculture_3pattern_full_logs_v0_2.ipynb)

Drive Workbenchの参考資料収集ノート。研究本線とは分離し、3つの異なる試合パターンを再走してログ一式をDriveへ保存する。

- seed 6105: strong win candidate
- seed 6104: close candidate
- seed 6108: loss candidate
- seat: 0
- Current: g17_agent.agent
- Opponent: repo agent.agent

Pinned source:
- relation-flow-agent: b3160a9e82e727627fe6659a37f0983970d4fbd8
- kaggle-environments: b2405492c8403f6649f9317290f215e0290a2425

出力先: Drive Workbench｜検証装置 / Raw Logs｜生ログ / WB-0001

各seed: FULL.md / decision_log.jsonl.gz / env_steps.json.gz / g17_final_trace.json.gz / telemetry.json / result.json

Boundary: 参考資料。Policy採用・因果推論の根拠へ自動昇格させない。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install requests jsonschema

from pathlib import Path
import os, sys, json, gzip, copy, zipfile, urllib.request, subprocess, hashlib, datetime

WORKBENCH = Path('/content/drive/MyDrive/Drive Workbench｜検証装置')
RAW_ROOT = WORKBENCH / 'Raw Logs｜生ログ'
RUN_DIR = RAW_ROOT / 'WB-0001'
RUN_DIR.mkdir(parents=True, exist_ok=True)

BASE = Path('/content/drive_workbench_runtime')
BASE.mkdir(parents=True, exist_ok=True)

RF_SHA = 'b3160a9e82e727627fe6659a37f0983970d4fbd8'
KE_SHA = 'b2405492c8403f6649f9317290f215e0290a2425'
RF_ZIP = BASE / f'relation-flow-agent-{RF_SHA}.zip'
KE_ZIP = BASE / f'kaggle-environments-{KE_SHA}.zip'

if not RF_ZIP.exists():
    urllib.request.urlretrieve(f'https://github.com/kw0809suzuki-oss/relation-flow-agent/archive/{RF_SHA}.zip', RF_ZIP)
if not KE_ZIP.exists():
    urllib.request.urlretrieve(f'https://github.com/Kaggle/kaggle-environments/archive/{KE_SHA}.zip', KE_ZIP)

RF_DIR = BASE / f'relation-flow-agent-{RF_SHA}'
KE_DIR = BASE / f'kaggle-environments-{KE_SHA}'

if not RF_DIR.exists():
    with zipfile.ZipFile(RF_ZIP) as z: z.extractall(BASE)
if not KE_DIR.exists():
    with zipfile.ZipFile(KE_ZIP) as z: z.extractall(BASE)

print('Workbench:', WORKBENCH)
print('Run dir  :', RUN_DIR)
print('RF SHA   :', RF_SHA)
print('KE SHA   :', KE_SHA)


In [ ]:
runner_source = r'''
import os, sys, json, gzip, copy, hashlib
from pathlib import Path

seed = int(sys.argv[1])
seat = int(sys.argv[2])
out_dir = Path(sys.argv[3])
rf_dir = Path(sys.argv[4])
ke_dir = Path(sys.argv[5])

sys.path.insert(0, str(ke_dir))
sys.path.insert(0, str(rf_dir))

from kaggle_environments import make
import g17_agent as current
import agent as opponent

ENV_FLAGS = {
    "ORIGIN_GATE_POLARITY":"inverted",
    "ORIGIN_GATE_MAGNITUDE":"0.04",
    "G15_CONNECT_OPPONENT_FIELD_DESCRIPTION":"1",
    "G15_ADAPTIVE_W_AMPLITUDE":"1",
    "G15_REMOVE_R_RELATION":"0",
    "G15_REMOVE_E_RELATION":"0",
    "G15_REMOVE_W_RELATION":"0",
    "G15_DISABLE_RESONANCE_CONTROL":"0",
    "ORIGIN_CROP_COMMITMENT":"1",
}
for k,v in ENV_FLAGS.items(): os.environ[k] = v

def safe_call(obj, name, *args):
    fn = getattr(obj, name, None)
    return fn(*args) if callable(fn) else None

def count_cows(obs):
    me = obs["farms"][obs["player"]]
    private = obs.get("private", {})
    shed = private.get("shed", {}) if isinstance(private, dict) else {}
    cows = shed.get("COW", 0) if isinstance(shed, dict) else 0
    try:
        for row in me.get("tiles", []):
            for tile in row:
                if isinstance(tile, dict) and tile.get("animal") == "COW": cows += 1
    except Exception:
        pass
    return int(cows)

def observer_snapshot(trace):
    if not isinstance(trace, dict): return {}
    body = trace.get("observe", {}).get("body", {})
    if not isinstance(body, dict): return {}
    out = {}
    for k in ["flip_events","semantic_reversal_events","boundary_events"]:
        arr = body.get(k)
        if isinstance(arr, list) and arr: out[k+"_latest"] = copy.deepcopy(arr[-1])
    for k in ["mode","target","feed","origin","origin_strategy","opponent_phase","distortion","counter_opportunity","counter_weight","resonance","probe_state"]:
        if k in body: out[k] = copy.deepcopy(body[k])
    return out

safe_call(current, "set_probe_enabled", True)
safe_call(current, "set_attribution_enabled", True)
safe_call(current, "reset_telemetry")
safe_call(current, "reset_trace")
safe_call(opponent, "reset_telemetry")
safe_call(opponent, "reset_trace")

decision_log = []
def observed(obs):
    me = obs["farms"][obs["player"]]
    action = current.agent(obs)
    trace = safe_call(current, "get_trace")
    decision_log.append({
        "turn": len(decision_log),
        "day": int(obs.get("day", -1)),
        "hour": int(obs.get("hour", -1)) if obs.get("hour") is not None else None,
        "player": int(obs.get("player", seat)),
        "money": float(me.get("money", 0)),
        "hands": len(me.get("hands", [])),
        "land": len(me.get("unlocked_quadrants", [])),
        "cows_observed": count_cows(obs),
        "observation_full": copy.deepcopy(obs),
        "action_full": copy.deepcopy(action),
        "observer_snapshot": observer_snapshot(trace),
    })
    return action

players = [opponent.agent, opponent.agent]
players[seat] = observed
env = make("kaggriculture", configuration={"seed": seed}, debug=False)
env.run(players)

rewards = [float(s.reward) for s in env.state]
terminal_self = rewards[seat]
terminal_opponent = rewards[1-seat]
margin = terminal_self - terminal_opponent
label = "WIN_STRONG" if margin >= 10000 else ("CLOSE" if abs(margin) <= 1500 else ("LOSS" if margin < 0 else "WIN"))
prefix = f"seed_{seed}_{label}"
telemetry = safe_call(current, "get_telemetry")
final_trace = safe_call(current, "get_trace")

def dump_json_gz(path, obj):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, default=str)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

decision_path = out_dir / f"{prefix}_decision_log.jsonl.gz"
with gzip.open(decision_path, "wt", encoding="utf-8") as f:
    for row in decision_log:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

env_steps_path = out_dir / f"{prefix}_env_steps.json.gz"
dump_json_gz(env_steps_path, env.steps)
trace_path = out_dir / f"{prefix}_g17_final_trace.json.gz"
dump_json_gz(trace_path, final_trace)
telemetry_path = out_dir / f"{prefix}_telemetry.json"
telemetry_path.write_text(json.dumps(telemetry, ensure_ascii=False, default=str, indent=2), encoding="utf-8")

md_path = out_dir / f"{prefix}_FULL.md"
with md_path.open("w", encoding="utf-8") as f:
    f.write(f"# Kaggriculture Full Decision Log | seed {seed} | {label}\n\n")
    f.write(f"- seat: {seat}\n- current: g17_agent.agent\n- opponent: repo agent.agent\n")
    f.write(f"- terminal_self: {terminal_self:.0f}\n- terminal_opponent: {terminal_opponent:.0f}\n- margin: {margin:+.0f}\n")
    f.write(f"- win: {terminal_self > terminal_opponent}\n- decision_turns: {len(decision_log)}\n")
    f.write("- raw complete trajectory: companion env_steps.json.gz\n")
    f.write("- full observation per decision: companion decision_log.jsonl.gz\n")
    f.write("- boundary: reference material only; not policy evidence\n\n## All decision turns\n\n")
    for row in decision_log:
        f.write(f"### Turn {row['turn']:03d} | Day {row['day']} | money {row['money']:.0f} | COW {row['cows_observed']} | HANDS {row['hands']} | LAND {row['land']}\n\n")
        f.write("Action JSON:\n\n")
        f.write(json.dumps(row["action_full"], ensure_ascii=False, default=str, indent=2))
        f.write("\n\nObserver snapshot JSON:\n\n")
        f.write(json.dumps(row["observer_snapshot"], ensure_ascii=False, default=str, indent=2))
        f.write("\n\n")

result = {
    "seed": seed, "seat": seat, "label": label,
    "terminal_self": terminal_self, "terminal_opponent": terminal_opponent,
    "margin": margin, "win": terminal_self > terminal_opponent,
    "decision_turns": len(decision_log), "environment_flags": ENV_FLAGS, "files": {}
}
for p in [md_path, decision_path, env_steps_path, trace_path, telemetry_path]:
    result["files"][p.name] = {"bytes": p.stat().st_size, "sha256": sha256(p)}

result_path = out_dir / f"{prefix}_result.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False))
'''
RUNNER = BASE / 'wb0001_runner.py'
RUNNER.write_text(runner_source, encoding='utf-8')
print('Runner:', RUNNER)


In [ ]:
SEEDS = [6105, 6104, 6108]
SEAT = 0
results = []

for seed in SEEDS:
    print(f"\n=== seed {seed} ===")
    cp = subprocess.run(
        [sys.executable, str(RUNNER), str(seed), str(SEAT), str(RUN_DIR), str(RF_DIR), str(KE_DIR)],
        text=True, capture_output=True, check=True
    )
    if cp.stderr.strip(): print(cp.stderr[-3000:])
    result = json.loads(cp.stdout.strip().splitlines()[-1])
    results.append(result)
    print("terminal:", result["terminal_self"], "vs", result["terminal_opponent"], "margin", result["margin"], "turns", result["decision_turns"])

manifest = {
    "run_id": "WB-0001",
    "purpose": "3-pattern full-log reference set",
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "relation_flow_agent_commit": RF_SHA,
    "kaggle_environments_commit": KE_SHA,
    "seat": SEAT, "seeds": SEEDS, "results": results,
    "boundary": "Reference material only; do not auto-promote to policy or causal evidence."
}
(RUN_DIR / "WB-0001_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

with (RUN_DIR / "INDEX.md").open("w", encoding="utf-8") as f:
    f.write("# WB-0001 | Kaggriculture 3-pattern full-log set\n\n")
    f.write(f"- relation-flow-agent commit: {RF_SHA}\n- kaggle-environments commit: {KE_SHA}\n- seat: {SEAT}\n")
    f.write("- boundary: reference material only; not policy evidence\n\n")
    f.write("| seed | label | self | opponent | margin | turns |\n|---:|---|---:|---:|---:|---:|\n")
    for r in results:
        f.write(f"| {r['seed']} | {r['label']} | {r['terminal_self']:.0f} | {r['terminal_opponent']:.0f} | {r['margin']:+.0f} | {r['decision_turns']} |\n")
    f.write("\nEach seed contains FULL.md + raw decision JSONL + raw env.steps + G17 trace + telemetry + result metadata.\n")

print("\nDONE")
print("Saved:", RUN_DIR)
